# Projeto Visão Computacional — Fase 02
## Reconhecimento da expressão facial da pessoa numa imagem

**Disciplina:** Visão Computacional (CCOM0034) — Prof. Geraldo Braz Junior
**Programa:** PPGCC — UFMA / NCA

**Equipe (Grupo 02):** Wesley Gatinho, Yuri (Yure) Torres, Osvaldo, Arnaud, Guilherme

**Repositório:** https://github.com/wesleygatinho/visao-visao_visao
**Dashboard de treino (Weights & Biases):** https://wandb.ai/wesley-gatinho-universidade-federal-do-maranh-o/expressao-facial-anynet-v2

---

Este notebook apresenta o pipeline completo da Fase 2 do projeto, cobrindo as etapas exigidas no enunciado:

1. CNN no espaço de projeto **AnyNet/RegNet** (d2l.ai)
2. Otimização de arquitetura e hiperparâmetros com **Optuna**
3. Registro dos experimentos no **Weights & Biases**
4. Aplicação de **webcam** (YOLO para detecção de rosto + AnyNet para a expressão)
5. *(Opcional)* Explicabilidade com **Grad-CAM**


## 1. Objetivo e continuidade em relação à Fase 1

A Fase 1 tratou apenas de **detectar se há pessoas** numa imagem. A Fase 2 dá continuidade
partindo do princípio de que os **rostos já foram detectados**, e o problema passa a ser:
dado um rosto recortado, **classificar a expressão facial** em uma de 8 categorias do
dataset FER+: `anger, contempt, disgust, fear, happy, neutral, sad, surprise`.

Diferente da Fase 1 (onde a arquitetura é escolhida "de prateleira" via transfer learning),
aqui a exigência é **projetar o espaço de arquiteturas** (AnyNet) e deixar uma busca
automatizada (Optuna) decidir a melhor configuração dentro desse espaço — o que justifica
a estrutura do código a seguir.


## 2. Arquitetura: CNN no espaço de projeto AnyNet

Seguindo https://d2l.ai/chapter_convolutional-modern/cnn-design.html, a rede é dividida em:

- **STEM**: convolução inicial que reduz a resolução pela metade
- **BODY**: sequência de estágios, cada um empilhando blocos **ResNeXt** (bottleneck + convolução agrupada)
- **HEAD**: global average pooling + classificador linear

A arquitetura inteira é parametrizada por `stem_channels`, `depths` (blocos por estágio),
`widths` (canais por estágio), `group_width` e `bot_mul` — esses são exatamente os graus
de liberdade que o Optuna varre na etapa de otimização.


In [ ]:
import sys, json
sys.path.insert(0, ".")

from models.anynet import AnyNet, ResNeXtBlock

# Exemplo de instanciação manual (arquitetura arbitrária, só para ilustrar o espaço de projeto)
exemplo = AnyNet(stem_channels=24, depths=[1, 2, 2], widths=[32, 64, 128],
                 group_width=8, bot_mul=1.0, num_classes=8, in_channels=1)
print(exemplo)
print(f"\nParâmetros treináveis: {exemplo.num_params():,}")


## 3. Otimização de arquitetura e hiperparâmetros com Optuna

O `optimize.py` roda uma *study* do Optuna (`TPESampler` + `MedianPruner`) que busca,
simultaneamente:

**Espaço de arquitetura (AnyNet):**
- `n_stages` ∈ [2, 4]
- `stem_channels` ∈ {16, 24, 32}
- `w0`, `wmult` → geram as `widths` de cada estágio (estilo RegNet, larguras crescentes)
- `depth_i` ∈ [1, 3] por estágio
- `group_width` ∈ {4, 8, 16}
- `bot_mul` ∈ {0.5, 1.0}

**Hiperparâmetros de treino:**
- `lr` ∈ [1e-4, 5e-3] (escala log)
- `weight_decay` ∈ [1e-5, 1e-3] (escala log)
- `optimizer` ∈ {adamw, sgd}

Cada trial é logado como uma run no W&B (grupo `optuna`). Ao todo foram avaliadas
**40 configurações** (39 trials + a run final de retreino), registradas no dashboard.


In [ ]:
with open("checkpoints/best_params.json") as f:
    best = json.load(f)

params = best["params"]
widths = best["widths"]
depths = [params[f"depth_{i}"] for i in range(params["n_stages"])]

print("== Melhor configuração encontrada pelo Optuna ==")
print(f"val_acc (curta, {12} épocas de busca): {best['value']:.4f}\n")

import pandas as pd
df = pd.DataFrame({
    "hiperparâmetro": list(params.keys()) + ["widths"],
    "valor": list(params.values()) + [widths],
})
df


In [ ]:
# Reconstrói a arquitetura vencedora a partir do best_params.json
melhor_modelo = AnyNet(
    stem_channels=params["stem_channels"],
    depths=depths,
    widths=widths,
    group_width=params["group_width"],
    bot_mul=params["bot_mul"],
    num_classes=8,
    in_channels=1,
)
print(melhor_modelo)
print(f"\nParâmetros treináveis: {melhor_modelo.num_params():,}")


### 3.1 Comparação entre trials do Optuna

Todos os **30 trials** da busca foram registrados no W&B, junto com o retreino final
(`gentle-lion-32`), e podem ser inspecionados diretamente no dashboard público do projeto:

**https://wandb.ai/wesley-gatinho-universidade-federal-do-maranh-o/expressao-facial-anynet-v2**

Na tabela de runs é possível ordenar por `val_acc` e comparar o `lr`, os `widths`, a
profundidade e as demais escolhas de cada configuração — o que mostra *por que* a
configuração vencedora foi escolhida, e não só qual foi ela.


**Leitura:** as trials rodam com um orçamento curto de épocas — o objetivo da busca é
apenas *ranquear* configurações, não treiná-las até o fim. Nas curvas do dashboard,
as trials com `lr` mal calibrado convergem visivelmente pior (a `val_acc` estaciona
cedo), enquanto as configurações estáveis se agrupam na faixa superior. A run final
(`gentle-lion-32`) se destaca das demais porque retreina a configuração vencedora com
o orçamento completo (até 100 épocas, com early stopping) — o que explica a distância
entre ela e as trials nas curvas de `val_acc`.


## 4. Retreino final da melhor arquitetura

A configuração vencedora do Optuna foi retreinada via `train.py` com um orçamento de até
**100 épocas** e **early stopping** (paciência de 10 épocas sem melhora na `val_acc`) —
uma melhoria em relação à versão anterior do experimento, que usava um número fixo de
60 épocas. O logging completo está no W&B (run `gentle-lion-32`, no projeto
`expressao-facial-anynet-v2`).

O early stopping interrompeu o treino na **época 77/100**: a melhor `val_acc` (0.7492)
foi atingida na época 67 e não houve melhora nas 10 épocas seguintes. Além de economizar
computação, isso garante que o checkpoint salvo corresponde ao ponto de melhor
generalização, e não ao final arbitrário do treino.

Abaixo, o log bruto de treino (`output.log`) é parseado e as curvas de treino/validação
são reconstruídas diretamente dos números — sem depender de nenhuma imagem externa.


### Log bruto de treino (`output.log`, run `gentle-lion-32`, early stopping na época 77/100)

```
[001/100] train_loss=1.9401 acc=0.2612 | val_loss=1.9027 acc=0.3146
[002/100] train_loss=1.6681 acc=0.4207 | val_loss=1.6326 acc=0.4453
[003/100] train_loss=1.4729 acc=0.5178 | val_loss=1.4734 acc=0.5278
[004/100] train_loss=1.3555 acc=0.5738 | val_loss=1.3393 acc=0.5945
[005/100] train_loss=1.2685 acc=0.6171 | val_loss=1.4399 acc=0.5550
[006/100] train_loss=1.1995 acc=0.6508 | val_loss=1.2934 acc=0.6196
[007/100] train_loss=1.1475 acc=0.6744 | val_loss=1.2456 acc=0.6516
[008/100] train_loss=1.0987 acc=0.6960 | val_loss=1.2896 acc=0.6202
[009/100] train_loss=1.0662 acc=0.7142 | val_loss=1.2728 acc=0.6113
[010/100] train_loss=1.0408 acc=0.7242 | val_loss=1.2035 acc=0.6697
[011/100] train_loss=1.0096 acc=0.7393 | val_loss=1.1848 acc=0.6692
[012/100] train_loss=0.9879 acc=0.7515 | val_loss=1.1885 acc=0.6613
[013/100] train_loss=0.9693 acc=0.7595 | val_loss=1.1701 acc=0.6809
[014/100] train_loss=0.9510 acc=0.7680 | val_loss=1.2624 acc=0.6345
[015/100] train_loss=0.9338 acc=0.7761 | val_loss=1.2227 acc=0.6516
[016/100] train_loss=0.9206 acc=0.7807 | val_loss=1.1940 acc=0.6762
[017/100] train_loss=0.9090 acc=0.7885 | val_loss=1.1789 acc=0.6865
[018/100] train_loss=0.9009 acc=0.7901 | val_loss=1.1643 acc=0.6787
[019/100] train_loss=0.8847 acc=0.8003 | val_loss=1.1562 acc=0.6888
[020/100] train_loss=0.8758 acc=0.8035 | val_loss=1.1446 acc=0.6916
[021/100] train_loss=0.8682 acc=0.8076 | val_loss=1.1416 acc=0.6952
[022/100] train_loss=0.8584 acc=0.8108 | val_loss=1.1687 acc=0.6821
[023/100] train_loss=0.8496 acc=0.8156 | val_loss=1.1768 acc=0.6784
[024/100] train_loss=0.8400 acc=0.8212 | val_loss=1.1530 acc=0.6921
[025/100] train_loss=0.8344 acc=0.8229 | val_loss=1.1345 acc=0.7067
[026/100] train_loss=0.8237 acc=0.8287 | val_loss=1.1264 acc=0.7095
[027/100] train_loss=0.8166 acc=0.8310 | val_loss=1.1417 acc=0.7058
[028/100] train_loss=0.8113 acc=0.8342 | val_loss=1.1211 acc=0.7170
[029/100] train_loss=0.8050 acc=0.8360 | val_loss=1.1438 acc=0.7067
[030/100] train_loss=0.7984 acc=0.8402 | val_loss=1.1343 acc=0.7067
[031/100] train_loss=0.7923 acc=0.8447 | val_loss=1.1050 acc=0.7263
[032/100] train_loss=0.7877 acc=0.8452 | val_loss=1.1299 acc=0.7218
[033/100] train_loss=0.7792 acc=0.8492 | val_loss=1.1142 acc=0.7319
[034/100] train_loss=0.7730 acc=0.8521 | val_loss=1.1464 acc=0.7033
[035/100] train_loss=0.7682 acc=0.8555 | val_loss=1.1042 acc=0.7319
[036/100] train_loss=0.7624 acc=0.8583 | val_loss=1.1258 acc=0.7204
[037/100] train_loss=0.7623 acc=0.8587 | val_loss=1.1007 acc=0.7246
[038/100] train_loss=0.7544 acc=0.8613 | val_loss=1.1497 acc=0.7011
[039/100] train_loss=0.7496 acc=0.8647 | val_loss=1.1129 acc=0.7277
[040/100] train_loss=0.7458 acc=0.8656 | val_loss=1.1392 acc=0.7114
[041/100] train_loss=0.7384 acc=0.8686 | val_loss=1.1163 acc=0.7254
[042/100] train_loss=0.7351 acc=0.8716 | val_loss=1.1189 acc=0.7327
[043/100] train_loss=0.7331 acc=0.8715 | val_loss=1.1222 acc=0.7154
[044/100] train_loss=0.7267 acc=0.8758 | val_loss=1.1534 acc=0.7131
[045/100] train_loss=0.7249 acc=0.8761 | val_loss=1.1077 acc=0.7268
[046/100] train_loss=0.7177 acc=0.8809 | val_loss=1.1032 acc=0.7333
[047/100] train_loss=0.7108 acc=0.8835 | val_loss=1.1219 acc=0.7240
[048/100] train_loss=0.7102 acc=0.8835 | val_loss=1.1174 acc=0.7240
[049/100] train_loss=0.7036 acc=0.8870 | val_loss=1.1253 acc=0.7313
[050/100] train_loss=0.7027 acc=0.8870 | val_loss=1.1020 acc=0.7352
[051/100] train_loss=0.6972 acc=0.8899 | val_loss=1.1178 acc=0.7296
[052/100] train_loss=0.6939 acc=0.8918 | val_loss=1.1309 acc=0.7238
[053/100] train_loss=0.6908 acc=0.8937 | val_loss=1.1250 acc=0.7308
[054/100] train_loss=0.6853 acc=0.8964 | val_loss=1.1086 acc=0.7350
[055/100] train_loss=0.6818 acc=0.8982 | val_loss=1.1197 acc=0.7333
[056/100] train_loss=0.6795 acc=0.8998 | val_loss=1.1158 acc=0.7333
[057/100] train_loss=0.6766 acc=0.9016 | val_loss=1.1206 acc=0.7330
[058/100] train_loss=0.6708 acc=0.9033 | val_loss=1.1251 acc=0.7308
[059/100] train_loss=0.6658 acc=0.9057 | val_loss=1.1111 acc=0.7439
[060/100] train_loss=0.6637 acc=0.9070 | val_loss=1.1089 acc=0.7459
[061/100] train_loss=0.6639 acc=0.9068 | val_loss=1.1101 acc=0.7408
[062/100] train_loss=0.6589 acc=0.9092 | val_loss=1.1221 acc=0.7397
[063/100] train_loss=0.6565 acc=0.9100 | val_loss=1.1158 acc=0.7431
[064/100] train_loss=0.6520 acc=0.9126 | val_loss=1.1155 acc=0.7296
[065/100] train_loss=0.6524 acc=0.9124 | val_loss=1.1057 acc=0.7420
[066/100] train_loss=0.6483 acc=0.9139 | val_loss=1.1148 acc=0.7366
[067/100] train_loss=0.6458 acc=0.9161 | val_loss=1.1050 acc=0.7492
[068/100] train_loss=0.6399 acc=0.9206 | val_loss=1.1216 acc=0.7478
[069/100] train_loss=0.6397 acc=0.9192 | val_loss=1.1180 acc=0.7445
[070/100] train_loss=0.6352 acc=0.9218 | val_loss=1.1147 acc=0.7439
[071/100] train_loss=0.6332 acc=0.9227 | val_loss=1.1248 acc=0.7464
[072/100] train_loss=0.6308 acc=0.9237 | val_loss=1.1259 acc=0.7411
[073/100] train_loss=0.6313 acc=0.9233 | val_loss=1.1194 acc=0.7445
[074/100] train_loss=0.6287 acc=0.9257 | val_loss=1.1276 acc=0.7422
[075/100] train_loss=0.6249 acc=0.9274 | val_loss=1.1247 acc=0.7406
[076/100] train_loss=0.6226 acc=0.9276 | val_loss=1.1219 acc=0.7448
[077/100] train_loss=0.6209 acc=0.9296 | val_loss=1.1134 acc=0.7450
early stopping na epoca 77/100 (sem melhora por 10 epocas, best_acc=0.7492)
melhor val_acc: 0.7492  ->  ./checkpoints/anynet_ferplus.pt
```


In [ ]:
import re
import matplotlib.pyplot as plt

rows = []
pattern = re.compile(
    r"\[(\d+)/(\d+)\] train_loss=([\d.]+) acc=([\d.]+) \| val_loss=([\d.]+) acc=([\d.]+)")

with open("output.log") as f:
    for line in f:
        m = pattern.match(line.strip())
        if m:
            ep, total, tl, ta, vl, va = m.groups()
            rows.append(dict(epoch=int(ep), total=int(total),
                             train_loss=float(tl), train_acc=float(ta),
                             val_loss=float(vl), val_acc=float(va)))

hist = pd.DataFrame(rows)
melhor_epoca = hist.loc[hist["val_acc"].idxmax()]
ultima_epoca = int(hist.epoch.max())
orcamento = int(hist.total.iloc[0])

print(f"Treino interrompido por early stopping na época {ultima_epoca}/{orcamento}")
print(f"Melhor época: {int(melhor_epoca.epoch)}  |  val_acc = {melhor_epoca.val_acc:.4f}")
hist.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(hist.epoch, hist.train_loss, label="train_loss")
axes[0].plot(hist.epoch, hist.val_loss, label="val_loss")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss — gentle-lion-32"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(hist.epoch, hist.train_acc, label="train_acc")
axes[1].plot(hist.epoch, hist.val_acc, label="val_acc")
axes[1].axvline(melhor_epoca.epoch, color="gray", linestyle="--", alpha=.6,
                label=f"melhor época ({int(melhor_epoca.epoch)})")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Acurácia")
axes[1].set_title("Acurácia — gentle-lion-32"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()


**Leitura dos resultados:**

- Convergência estável: acurácia de treino sai de ~26% e chega a ~93% até a interrupção
- Melhor checkpoint salvo na **época 67**, com `val_acc = 0.7492`
- O **early stopping** disparou na época 77 (10 épocas sem melhora na validação), evitando
  23 épocas de treino que só aumentariam o gap treino–validação sem ganho real
- A `val_loss` estabiliza em torno de ~1.10–1.12 a partir da época ~30, enquanto a
  `train_loss` continua caindo — sinal clássico de que o modelo esgotou o que consegue
  extrair da validação, exatamente o cenário que o early stopping existe para cortar
- Gap treino–validação final (~0.93 vs ~0.75) é esperado para o FER+, que tem ambiguidade
  inerente entre classes de expressão (ex.: *neutral* vs *sad* sutil)


## 5. Registro no Weights & Biases

Prints do dashboard oficial do projeto (`expressao-facial-anynet-v2`), gerados diretamente
pela plataforma — confirmam que o tracking foi feito de ponta a ponta, e batem com os
números recalculados acima a partir do `output.log`.

![val_loss](assets/val_loss.png)
![val_acc](assets/val_acc.png)
![train_loss](assets/train_loss.png)
![train_acc](assets/train_acc.png)
![lr](assets/lr.png)
![epoch](assets/epoch.png)

Dashboard completo (31 runs — 30 trials do Optuna + retreino final):
https://wandb.ai/wesley-gatinho-universidade-federal-do-maranh-o/expressao-facial-anynet-v2

Run do retreino final (`gentle-lion-32`):
https://wandb.ai/wesley-gatinho-universidade-federal-do-maranh-o/expressao-facial-anynet-v2/runs/u5k4hrf2


## 6. Aplicação de webcam

Pipeline em `app/webcam_app.py`:

1. **Detecção de rosto**: YOLOv8-face (baixado automaticamente do Hugging Face); se falhar,
   usa Haar Cascade do OpenCV como fallback
2. Cada rosto detectado é recortado, convertido para tons de cinza e redimensionado para
   48×48 (mesmo pré-processamento do treino)
3. O **AnyNet** treinado classifica a expressão do recorte
4. O rótulo é desenhado na *bounding box* em tempo real

```bash
python app/webcam_app.py --ckpt checkpoints/anynet_ferplus.pt
```


In [ ]:
# Célula de demonstração — só executa se RUN_DEMO=True
# (mantido False por padrão para não travar a correção automática do notebook)
RUN_DEMO = False

if RUN_DEMO:
    import subprocess
    subprocess.run(["python", "app/webcam_app.py", "--ckpt", "checkpoints/anynet_ferplus.pt"])
else:
    print("Demonstração ao vivo — ative RUN_DEMO=True durante a apresentação.")


## 7. [Opcional] Explicabilidade com Grad-CAM

`explain/gradcam.py` implementa Grad-CAM sobre o último estágio do `body` do AnyNet
(exposto via `model.last_conv_stage`), gerando um heatmap que mostra quais regiões do
rosto mais influenciaram a classificação da expressão. Pode ser ativado na aplicação de
webcam com a flag `--explain`:

```bash
python app/webcam_app.py --ckpt checkpoints/anynet_ferplus.pt --explain
```


## 8. Conclusões

- A busca com Optuna cobriu 30 configurações no espaço AnyNet/RegNet, com todos os trials
  rastreados no W&B, permitindo comparar arquiteturas e hiperparâmetros de forma sistemática
  em vez de tentativa e erro manual
- A arquitetura vencedora (3 estágios, `stem_channels=16`, `widths=[32,64,120]`, otimizador
  AdamW) atingiu **74,92% de acurácia de validação** no FER+ no retreino completo, com
  **early stopping** (paciência de 10) interrompendo o treino na época 77/100 — o checkpoint
  final corresponde ao ponto de melhor generalização, não a um corte arbitrário
- O pipeline fecha o ciclo completo pedido no enunciado: dataset → arquitetura parametrizável
  → busca automatizada → tracking de experimentos → aplicação funcional em tempo real
- Trabalhos futuros: aumentar o espaço de busca do Optuna (mais trials, mais épocas por
  trial), e comparar contra uma segunda rota de detecção de rosto além do YOLO

---

**Repositório completo:** https://github.com/wesleygatinho/visao-visao_visao
**Dashboard W&B:** https://wandb.ai/wesley-gatinho-universidade-federal-do-maranh-o/expressao-facial-anynet-v2
